# Table 2 -  Machine Learning Classifier Selection 

Performance summary of machine learning classifiers of F1||F5 delta radiomics model on internal adrenal patients with Lung and Pancreas Auxiliary Data.

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=RuntimeWarning)

# import Libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV,ParameterGrid
from sklearn.linear_model import Lasso, Ridge, ElasticNet
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn import metrics
import xgboost as xgb
from sklearn.metrics import mean_squared_error
from sklearn.metrics import confusion_matrix,accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,classification_report
from sklearn.model_selection import KFold,cross_val_score,cross_validate, cross_val_predict,RepeatedKFold, StratifiedKFold,RepeatedStratifiedKFold
from sklearn.pipeline import make_pipeline
from random import randrange, uniform
from imblearn.over_sampling import SMOTE
from imblearn.over_sampling import ADASYN
from imblearn.over_sampling import BorderlineSMOTE,SVMSMOTE,SMOTEN
from sklearn.neighbors import NearestNeighbors
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
# Load Dataset

AdrenalData = pd.read_excel("Path to Internal Adrenal Dataset")
LungData = pd.read_excel("Path to Internal Lung Dataset")
PancreasData = pd.read_excel("Path to Internal Pancreas Dataset")

# Define the target and feature vectors in the dataset

Adrenal_Ytrain = AdrenalData.iloc[:,0] # use the slicing value that applies to your data
Adrenal_Xtrain = AdrenalData.iloc[:,1:]

Lung_Ytrain = LungData.iloc[:,1]
Lung_Xtrain = LungData[:,2:]


pancreas_Ytrain = PancreasData.iloc[:,1]
pancreas_Xtrain = PancreasData.iloc[:,2:]


In [ ]:
# functions

# Bootstrap Function
def Bootstrap(x, y, n_split, random_seed=42):
    import numpy as np
    np.random.seed(42)
    rng = np.random.RandomState(random_seed)
    sample_idx = np.arange(x.shape[0])
    set_idx = set(sample_idx)
    Train_id={}
    Test_id={}
    for i in range(n_split):
        while True:
            train_idx= rng.choice(sample_idx[y==1],
                                   size=np.sum(y),
                                   replace=True)
            train_idx= np.append(train_idx, rng.choice(sample_idx[y==0],
                                   size=np.sum(1-y),
                                   replace=True))
            
            if len(set(sample_idx[y==1]).difference(set(train_idx))) and len(set(sample_idx[y==0]).difference(set(train_idx))):
                break
        test_idx = np.array(list(set_idx - set(train_idx)))
        Train_id[i]=train_idx
        Test_id[i]=test_idx
            
    return Train_id, Test_id  
     

# Confidence Intervals Function

def mean_confidence_interval(data, confidence=0.95):
    import numpy as np
    import scipy.stats
    a = 1.0 * np.array(data)
    n = len(a)
    m, se = np.mean(a), scipy.stats.sem(a)
    h = se * 1.96
    
    return m-h, m+h

In [ ]:
# Parameter Grid for Feature Selection and Hyperparameter Tuning

# Feel free to modify

# ElasticNet Feature Selection Parameter Grid and Pipleline
param_grid_EN = {
    'model__alpha': [0.1],
    'model__l1_ratio':[0.4],
}

pipeline_EN = Pipeline([
            ('scaler', StandardScaler()),
            ('model', ElasticNet())
        ])


# Machine Learning Classifiers

classifiers = {
    "LR": LogisticRegression(),
    "NB": GaussianNB(),
    "SVC": SVC(),
    "KNN": KNeighborsClassifier(),
    "DT": DecisionTreeClassifier(),
    "RF": RandomForestClassifier(),     
    }

# Classifier Pipelines

pipelines = {}
for name, classifier in classifiers.items():
    pipeline = Pipeline([
        ("std", StandardScaler()),
        ("classifier", classifier)
    ])
    
    pipelines[name] = pipeline

# Parameter grids for classifiers
params = {
    "DT": {
        'classifier__max_depth': np.arange(2, 10, 1),
        'classifier__criterion': ['gini', 'entropy'],
        'classifier__min_samples_split': np.arange(2, 5, 1),
        'classifier__min_samples_leaf': np.arange(2, 5, 1),
        'classifier__random_state': [0,1,10,42],
        
    },
    
    "RF": {
        'classifier__max_depth': np.arange(3, 10, 1),
        'classifier__min_samples_split': np.arange(1, 10, 1),
        'classifier__min_samples_leaf': np.arange(1, 5, 1),
        'classifier__n_estimators': np.arange(5,30,5),
        'classifier__random_state': [1],
        
    },
    
    "LR": {
        'classifier__solver': ['liblinear', 'sag', 'saga', 'newton-cg', 'lbfgs'],
        'classifier__C': np.arange(0.01, 5, 0.01),
        'classifier__max_iter': [10, 25, 30, 50, 100, 200],
        
    },
    "NB": {
        'classifier__var_smoothing': [0,1e-9]
    },
    "SVC": {
        'classifier__kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
        'classifier__degree': np.arange(0, 3, 1),
        'classifier__gamma': ['scale', 'auto'],
        "classifier__C": np.arange(0.01, 4, 0.1),
        
    },
    
    "KNN": {
        'classifier__n_neighbors': np.arange(1, 10, 1),
        'classifier__weights': ['uniform', 'distance'],
        'classifier__p': np.arange(1, 4, 1)

}

}

# Model Development 





In [ ]:
# Divide the Adrenal Data into 42 Bootstrapped Train and Test folds

Train_id, Test_id = Bootstrap(Adrenal_Xtrain, Adrenal_Ytrain, n_split=42, random_seed=42)

# Define inner cross-validation fold

inner_kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

# Feature selection, Hyperparameter tuning and best estimator selection.

best_estimators = []  # List of the best estimators after hyperparameter tuning


for i in range(len(Train_id)):
    X_train, y_train = Adrenal_Xtrain.iloc[Train_id[i]], Adrenal_Ytrain.iloc[Train_id[i]]
    X_test, y_test = Adrenal_Xtrain.iloc[Test_id[i]], Adrenal_Ytrain.iloc[Test_id[i]]
    
    # Feature Selection
    # Note that feature selection is done only on the Adrenal Data and not on auxiliary data.
    search = GridSearchCV(pipeline_EN,
                      param_grid_EN,
                      cv=inner_kf,
                      scoring="neg_root_mean_squared_error")

    search.fit(X_train, y_train)

    # Get the best parameters
    best_parameter = search.best_params_
    coefficients = search.best_estimator_.named_steps['model'].coef_
    selected_features = np.array(features)[coefficients != 0]
    print(selected_features)
    print("####################################################################################")


    
    # Concatenate the Auxiliary Data after feature selection
    
    # Comment out the auxilary data you are not using
    
    # Pancreas
    XX_train = pd.concat([X_train, pancreas_Xtrain])
    yy_train = pd.concat([y_train, pancreas_Ytrain])
    
    # OR
    
    # Lung
    #XX_train = pd.concat([X_train, lung_Xtrain])
    #yy_train = pd.concat([y_train, lung_Ytrain])
    
    
    
    # Use the selected features for further processing
    Xtrain_hyper = XX_train[selected_features] # This is what is passed on for hyperparameter tuning
    Xvalid = X_test[selected_features] # This is the Adrenal Data that is reserved for testing the best estimator from the hyperparameter tuning.
    
    # Data Augmentation - BorderlineSMOTE-1 
    bsmote = BorderlineSMOTE(random_state=10, kind='borderline-1',k_neighbors=3, sampling_strategy='minority')
    X_resampled, y_resampled = bsmote.fit_resample(Xtrain_hyper, y_train)
    
    
    # Hyperparameter Tuning
    grid_kf = {}
    for name, pipeline in pipelines.items():
        grid_search_inner_kf = GridSearchCV(estimator=pipeline,
                                            param_grid=params[name],
                                            scoring="roc_auc",
                                            n_jobs=-1,
                                            cv=inner_kf,
                                            verbose=0,
                                            refit=True)
        grid_kf[name] = grid_search_inner_kf

    for name, grid_search in grid_kf.items():
        grid_kf[name].fit(X_resampled, y_resampled)
        best_estimator = grid_kf[name].best_estimator_
        ypred = grid_kf[name].best_estimator_.predict(Xvalid_hyper)
        
        # Check for empty and unique predictions
        if ypred is not None:  
            unique_classes = np.unique(ypred)
            if len(unique_classes) == 1:
                print(f"Skip: Estimator from '{name}' predicts only one class.")
                continue  # Skip to the next iteration

            score = roc_auc_score(ypred, y_test)
            print("estimator:", best_estimator)
            print('estimator_score: ', score)
            print("-" * 100)
            # Save the best estimator, its score and the selected features in the outer list- best estimator
        best_estimators.append({'Algorithm' : name, 'selected_features': selected_features, 'estimator': best_estimator, 'score': score})


# Model Evaluation 

In [ ]:
# Filter best estimators of each classifier's results
classifier_results = [estimator for estimator in best_estimators if estimator['Algorithm'] == 'Name of classifier']

# best estimator is the estimator with the highest AUC on the Adrenal validation fold
best_estimator = max(classifier_results, key=lambda x: x['score'])
best_features = best_estimator_score['selected_features']
clf = best_estimator['estimator']


# Model Evaluation 

model_kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

outer_scores = []
for train_idx, valid_idx in model_kf.split(Adrenal_Xtrain, Adrenal_Ytrain):
    X_train, X_valid = Adrenal_Xtrain.iloc[train_idx], Adrenal_Xtrain.iloc[valid_idx]
    y_train, y_valid = Adrenal_Ytrain.iloc[train_idx], Adrenal_Ytrain.iloc[valid_idx]
    
    # Concatenate Auxiliary Data
    X_train = pd.concat([X_train, lung_Xtrain])
    y_train = pd.concat([y_train, lung_Ytrain])
    
    # or
    
    #X_train = pd.concat([X_train, pancreas_Xtrain])
    #y_train = pd.concat([y_train, pancreas_Ytrain])
    

    bsmote = BorderlineSMOTE(random_state=10, kind='borderline-1',k_neighbors=3, sampling_strategy='minority')
    X_resampled, y_resampled = bsmote.fit_resample(X_train, y_train)

    clf.fit(X_resampled, y_resampled)
    y_pred = clf.predict(X_valid)
    AUC_X = roc_auc_score(y_valid,y_pred)
    outer_scores.append(AUC_X)
    
print(np.mean(outer_scores))
CI = mean_confidence_interval(outer_scores)
print(CI)
